**Start at run here**

In [ ]:
import pandas as pd
from utils.eda_utils import load_images, download_clip 
from transformers import AutoProcessor, pipeline
from optimum.onnxruntime import ORTModelForZeroShotImageClassification

print("Loading optimized ONNX model...")
model_id = "openai/clip-vit-base-patch32"

model = ORTModelForZeroShotImageClassification.from_pretrained(model_id, export=True)
processor = AutoProcessor.from_pretrained(model_id)

# initialize the clip model
classifier = pipeline(
    "zero-shot-image-classification",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.image_processor,
    provider="CPUExecutionProvider"
)

# sample labels
label_list = ["with a license plate", "without a license plate"]
color_list = ["white", "black", "red", "orange", "yellow", "green", "blue", "purple"]
vehicle_list = ["car", "bus", "motorcycle", "plate with many words", "train"]

# directory with images
image_folder = r"../data/license_plate_detection/train/images"


# load images from image directory
image_set = load_images(directory= image_folder, num_img= 100, use_rand= True, img_obj= True)

# get pillow image objects and file names from loaded images
pil_images = [img[0] for img in image_set]
file_names = [img[1] for img in image_set]

print("Running zero-shot classification via ONNX Runtime....")
# run results through clip classifier in batches
results = classifier(pil_images, candidate_labels=vehicle_list, batch_size=8)
    
prob_list = []

# iterate through results, assign probabilities
for idx, (predictions, file_name) in enumerate(zip(results, file_names)):
    df = pd.DataFrame(predictions).T
    df.columns = df.iloc[-1]
    df = df[:-1]
    df["fn"] = file_name
    prob_list.append(df)
    
    pct_complete = (idx + 1) / len(image_set)
    print(f"%{pct_complete*100:.2f} formatted")


In [ ]:
import pandas as pd
result = pd.concat(results, ignore_index=True)

result.to_json("C:/Users/Installer/Downloads/vehicle_labels.json")

df = pd.read_json("C:/Users/Installer/Downloads/vehicle_labels.json")

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
df_license_plate_only = df[df["white"] > .1].reset_index()

cols = 3
n = df_license_plate_only.shape[0]
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize = (12, rows * 3) )

axes_flat = axes.flatten()

for idx, row in df_license_plate_only.iterrows():
    axes_flat[idx].imshow(Image.open(row["fn"]))
    axes_flat[idx].set_title(f'Probability {row["white"]:.2f}')

**Run the cells below**

In [ ]:
import pandas as pd
import warnings
from torch.jit import TracerWarning
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from utils.zero_shot_utils import load_images, load_clip_pipeline, run_classification, prob_results

warnings.filterwarnings("ignore", category=TracerWarning)

In [ ]:
classifier = load_clip_pipeline()

# run as many times as you want with different labels
results = run_classification(
    classifier=classifier,
    label_list=["license plate", "american"],
)

p_results = prob_results(
    results, "C:/Users/Installer/Downloads/vehicle_labels.json", label="license plate", plot=True
)
print(p_results)